# OutputFixingParser → 예방(구조화 출력) + 명시적 복구

`OutputFixingParser`는 LangChain v1에서 **`langchain-classic`** 으로 이동한 레거시 API입니다. 또한 책의 `from langchain.output_parsers import PydanticOutputParser` 경로 역시 v1에서는 `langchain_core.output_parsers`를 사용해야 합니다.

현재 권장되는 접근은 두 단계입니다.

1. **예방**: `with_structured_output()`으로 애초에 스키마를 준수하는 출력을 생성한다. 공급자가 JSON Schema 준수를 강제하므로 형식 오류 자체가 크게 줄어듭니다.
2. **복구**: 그래도 형식이 깨진 텍스트를 다뤄야 한다면(외부 시스템 출력, 구조화 출력 미지원 모델 등)
   - 먼저 **LLM 호출 없는 결정적 복구**를 시도하고,
   - 안 되면 **"오류 메시지 + 원본"을 구조화 출력 모델에 넘겨 고치는 체인**을 직접 구성합니다. (`OutputFixingParser`가 내부에서 하던 일을 투명하게 작성하는 것)

> 에이전트(`create_agent`)에서 `response_format=ToolStrategy(Schema)`를 쓰면, 검증 실패 시 오류를 모델에 돌려주고 재시도하는 동작(`handle_errors`)이 기본으로 켜져 있습니다.

In [ ]:
# 최초 1회 설치 (LangChain v1 기준)
# %pip install -qU langchain langchain-openai langchain-classic python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # .env 의 OPENAI_API_KEY, LANGSMITH_API_KEY 를 불러옵니다.

# LangSmith 추적: 별도 헬퍼 없이 환경변수만 설정하면 자동으로 활성화됩니다.
if os.getenv("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ.setdefault("LANGSMITH_PROJECT", "CH03-OutputParser")

In [ ]:
from langchain.chat_models import init_chat_model

# 공급자 중립적인 모델 초기화 ("공급자:모델명")
# 다른 모델로 바꾸려면 문자열만 교체하면 됩니다. 예) "anthropic:claude-sonnet-4-5", "ollama:llama3.1"
llm = init_chat_model("openai:gpt-4.1-mini", temperature=0)

In [ ]:
from langchain_core.exceptions import OutputParserException
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field


class Actor(BaseModel):
    """배우와 출연작 목록"""

    name: str = Field(description="name of an actor")
    film_names: list[str] = Field(description="list of names of films they starred in")


actor_query = "Generate the filmography for a random actor."

parser = PydanticOutputParser(pydantic_object=Actor)

## 1. 문제 상황 재현

작은따옴표를 쓴 파이썬 dict 표기는 유효한 JSON이 아니므로 파싱에 실패합니다. 이번에는 예외를 잡아서 오류 메시지를 확인합니다.

In [ ]:
misformatted = "{'name': 'Tom Hanks', 'film_names': ['Forrest Gump']}"

try:
    parser.parse(misformatted)
except OutputParserException as e:
    print("파싱 실패:", e)

## 2. 결정적 복구 (LLM 호출 없음)

이 예시처럼 원인이 명확한 오류는 LLM 없이도 고칠 수 있습니다. 비용이 없고 결과가 항상 같으므로 먼저 시도할 가치가 있습니다.
(`eval()`이 아닌 `ast.literal_eval()`은 리터럴만 해석하므로 안전합니다.)

In [ ]:
import ast

Actor.model_validate(ast.literal_eval(misformatted))

## 3. LLM 기반 복구 체인 (`OutputFixingParser`의 현대적 대체)

원본 출력과 오류 메시지를 함께 주고, **구조화 출력 모델**이 스키마에 맞는 결과를 직접 생성하게 합니다.
책의 `OutputFixingParser`는 "고친 텍스트"를 다시 파싱했지만, 여기서는 복구 결과가 곧바로 검증된 `Actor` 객체입니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

fix_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "아래 출력은 스키마에 맞지 않아 파싱에 실패했습니다. "
            "원래 내용을 바꾸지 말고 스키마에 맞게 고쳐 주세요.",
        ),
        ("human", "## 원본 출력\n{completion}\n\n## 오류\n{error}"),
    ]
)

fixer = fix_prompt | llm.with_structured_output(Actor)


def parse_or_fix(text: str) -> Actor:
    """먼저 일반 파싱을 시도하고, 실패하면 LLM으로 복구합니다."""
    try:
        return parser.parse(text)
    except OutputParserException as e:
        return fixer.invoke({"completion": text, "error": str(e)})


actor = parse_or_fix(misformatted)
actor

이 함수는 LCEL 체인의 한 단계로도 넣을 수 있습니다. 구조화 출력을 지원하지 않는 모델로 텍스트를 생성하는 경우의 파이프라인 예시입니다.

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

gen_prompt = ChatPromptTemplate.from_template("{query}\n\n{format_instructions}").partial(
    format_instructions=parser.get_format_instructions()
)

robust_chain = gen_prompt | llm | StrOutputParser() | RunnableLambda(parse_or_fix)
robust_chain.invoke({"query": actor_query})

## 4. 예방: 처음부터 구조화 출력으로 생성

대부분의 경우 이것만으로 충분합니다. 여기에 LCEL의 범용 안정성 도구를 조합할 수 있습니다.
- `.with_retry()` : 일시적 오류(네트워크, 레이트리밋 등) 시 재시도
- `.with_fallbacks([...])` : 실패 시 다른 모델/체인으로 대체
- `include_raw=True` : 예외 대신 `parsing_error`로 실패 여부를 확인

In [ ]:
structured_actor = llm.with_structured_output(Actor).with_retry(stop_after_attempt=3)

structured_actor.invoke(actor_query)

In [ ]:
checked = llm.with_structured_output(Actor, include_raw=True).invoke(actor_query)

if checked["parsing_error"] is None:
    print("성공:", checked["parsed"])
else:
    # 실패 시 원본 텍스트를 복구 체인으로 넘길 수 있습니다.
    print("실패:", checked["parsing_error"])
    print(parse_or_fix(checked["raw"].text))

## (참고) 레거시 API

기존 코드 유지보수가 목적이라면 다음과 같이 사용할 수 있습니다. 신규 코드에는 권장하지 않습니다.

```python
from langchain_classic.output_parsers import OutputFixingParser

new_parser = OutputFixingParser.from_llm(parser=parser, llm=llm)
new_parser.parse(misformatted)
```